In [6]:
from tabpfn.priors.mlp import get_batch
from tabpfn.priors.mlp_anti import get_batch_anti
import torch

In [7]:
hyperparameters = {
    "mix_activations": True,
    #"num_layers": {'distribution': 'meta_trunc_norm_log_scaled', 'max_mean': 6, 'min_mean': 1, 'round': True,
    #               'lower_bound': 2},
    "num_layers": {'distribution': 'meta_gamma', 'max_alpha': 2, 'max_scale': 3, 'round': True,
                    'lower_bound': 2},
    # Better beta?
    #"prior_mlp_hidden_dim": {'distribution': 'meta_trunc_norm_log_scaled', 'max_mean': 130, 'min_mean': 5,
    #                         'round': True, 'lower_bound': 4},
    "prior_mlp_hidden_dim": {'distribution': 'meta_gamma', 'max_alpha': 3, 'max_scale': 100, 'round': True, 'lower_bound': 4},

    "prior_mlp_dropout_prob": {'distribution': 'meta_beta', 'scale': 0.6, 'min': 0.1, 'max': 5.0},
# This mustn't be too high since activations get too large otherwise

    "noise_std": {'distribution': 'meta_trunc_norm_log_scaled', 'max_mean': .3, 'min_mean': 0.0001, 'round': False,
                    'lower_bound': 0.0},
    "init_std": {'distribution': 'meta_trunc_norm_log_scaled', 'max_mean': 10.0, 'min_mean': 0.01, 'round': False,
                    'lower_bound': 0.0},
    #"num_causes": {'distribution': 'meta_trunc_norm_log_scaled', 'max_mean': 12, 'min_mean': 1, 'round': True,
    #               'lower_bound': 1},
    "num_causes": {'distribution': 'meta_gamma', 'max_alpha': 3, 'max_scale': 7, 'round': True,
                                'lower_bound': 2},

    "is_causal": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    "pre_sample_weights": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    "y_is_effect": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    "sampling": {'distribution': 'meta_choice', 'choice_values': ['normal', 'mixed']},
    "prior_mlp_activations": {'distribution': 'meta_choice_mixed', 'choice_values': [
        torch.nn.Tanh
        , torch.nn.Identity
        , torch.nn.ReLU
    ]},
    "block_wise_dropout": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    "sort_features": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    "in_clique": {'distribution': 'meta_choice', 'choice_values': [True, False]},
    #'pre_sample_causes': {'distribution': 'meta_choice', 'choice_values': [True, False]},
}

In [ ]:
import numpy as np
import scipy.stats as stats
import random
import math

def sample_hyperparameters(hp_dict):
    sampled = {}
    for key, value in hp_dict.items():
        if isinstance(value, dict) and 'distribution' in value:
            dist = value['distribution']
            val = None
            if dist == 'meta_gamma':
                # Simplified meta_gamma logic based on uniform alpha & scale
                alpha = np.random.uniform(0.1, value.get('max_alpha', 1.0))
                scale = np.random.uniform(0.1, value.get('max_scale', 1.0))
                val = np.random.gamma(shape=alpha, scale=scale)
            elif dist == 'meta_beta':
                # Simplified meta_beta logic
                scale = value.get('scale', 1.0)
                min_val = value.get('min', 0.0)
                max_val = value.get('max', 2.0)
                a = np.random.uniform(0.1, max_val)
                b = np.random.uniform(0.1, max_val)
                val = min_val + np.random.beta(a, b) * (scale)
            elif dist == 'meta_trunc_norm_log_scaled':
                # Simplified meta_trunc_norm_log_scaled, scaling with logs
                min_mean = value.get('min_mean', 0.0001)
                max_mean = value.get('max_mean', 1.0)
                log_mean = np.random.uniform(math.log(min_mean), math.log(max_mean))
                mean = math.exp(log_mean)
                log_std = np.random.uniform(math.log(0.01), math.log(1.0))
                std = mean * math.exp(log_std)
                # Sample from truncated normal with lower bound 0
                val = float(stats.truncnorm((0 - mean) / std, (1000 - mean) / std, loc=mean, scale=std).rvs())
            elif dist in ['meta_choice', 'meta_choice_mixed']:
                val = random.choice(value['choice_values'])
            else:
                val = None

            if val is not None:
                if value.get('round', False):
                    val = int(np.round(val))
                if 'lower_bound' in value:
                    val = max(value['lower_bound'], val)

            sampled[key] = val
        else:
            sampled[key] = value

    return sampled

# Get our new sample directly
sampled_hyperparameters = sample_hyperparameters(hyperparameters)

# Provide defaults for missing parameters
defaults = {
    "pre_sample_causes": True,
    "prior_mlp_scale_weights_sqrt": True,
    "random_feature_rotation": False,
    "differentiable": True,
}
for k, v in defaults.items():
    if k not in sampled_hyperparameters:
        sampled_hyperparameters[k] = v


# To support the batch function which expects a callable for prior_mlp_activations
if "prior_mlp_activations" in sampled_hyperparameters:
    activation_class = sampled_hyperparameters["prior_mlp_activations"]
    sampled_hyperparameters["prior_mlp_activations"] = lambda: activation_class()

In [ ]:
# Process batch based on manually sampled values
x, y, y_ = get_batch(10, 20, 20, sampled_hyperparameters)
print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: torch.Size([20, 10, 20])
y shape: torch.Size([20, 10])


In [11]:
x_anti, y_anti, y__anti  = get_batch_anti(10, 20, 20, sampled_hyperparameters)
print("x shape:", x_anti.shape)
print("y shape:", y_anti.shape)

x shape: torch.Size([20, 10, 20])
y shape: torch.Size([20, 10])
